# HELM MAAF — explications selon le profil
Sources GitHub figées à 612b87c6856d3d41ea069b0243d162f033538e55. Quatre profils : utilisateur, métier, technique et audit. La proposition de méthodes est une règle explicite, sans apprentissage utilisateur. Même protocole MAAF que la recette v1.

## 1. Récupérer les codes sur GitHub
Dépôt privé : lien d’archive temporaire saisi masqué. Aucun jeton de compte n’est envoyé dans Colab. Un lien expiré doit être renouvelé.


**Version colorée 0.2.0a3** : quatre profils accessibles par onglets HTML autonomes ; les contributions restent signées et les méthodes distinctes.


In [ ]:
import hashlib
import io
import sys
import urllib.request
import zipfile
from pathlib import Path
from getpass import getpass

REVISION = "612b87c6856d3d41ea069b0243d162f033538e55"
if globals().get("ARCHIVE_REVISION") != REVISION:
    archive_url = getpass("Lien GitHub temporaire de la révision : ")
    assert archive_url.startswith("https://codeload.github.com/KossiFO/helm-xai-these/")
    archive = urllib.request.urlopen(archive_url, timeout=120).read()
    del archive_url
    ARCHIVE_REVISION = REVISION
source_dir = Path("/content/helm_github_source")
source_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as zipped:
    root_name = zipped.namelist()[0].split("/")[0]
    archive_revision = root_name.rsplit("-", 1)[-1]
    assert len(archive_revision) >= 7 and REVISION.startswith(archive_revision), "Révision inattendue"
    assert all((source_dir / n).resolve().is_relative_to(source_dir.resolve()) for n in zipped.namelist())
    zipped.extractall(source_dir)
SOURCE = source_dir / root_name
for module_name in list(sys.modules):
    if module_name == "helm_colab" or module_name.startswith("helm_colab."):
        del sys.modules[module_name]
sys.path.insert(0, str(SOURCE / "examples/colab"))
print("Codes récupérés depuis GitHub :", REVISION)
print("Archive SHA256 :", hashlib.sha256(archive).hexdigest())
print("Sources consultables dans le panneau Fichiers :", SOURCE)


## 2. Installer HELM dans Colab
Environnement Python réservé au tabulaire ; préparation technique séparée dans `helm_colab/session.py`.

In [ ]:
from helm_colab import ColabExperiment
experience = ColabExperiment()
experience.install()


## 3. Charger les données et afficher les variables
Échantillon du 25/08/2026, vérifié par SHA256. Le modèle est réentraîné dans Colab ; la cible, les scores préexistants et les colonnes Sentinelle sont exclus des prédicteurs. Les valeurs expliquées sont celles après préparation, notamment imputation apprise sur l’entraînement.

In [ ]:
from google.colab import files
if not Path("/content/sample_maaf_entrainement_20260825_095534.csv").exists():
    files.upload()
experience.prepare()


## 4. Expliquer la classe 1 avec SHAP
Les 10 plus hauts scores et les 10 plus faibles sont expliqués. La décision du modèle reste inchangée. Fond : 50 lignes d’entraînement, graine 42 ; budget : 512 coalitions. L’additivité est vérifiée, mais ne constitue pas une validation de fidélité globale.

Appel natif effectué par le module de calcul :
```python
from helm import TabularHELM
helm = TabularHELM(model, X_train, target_class=1, background_size=50, random_state=42)
rapport = helm.explain_top(X_test, k=10, order="highest", labels=y_test, method="shap", budget=512)
```

In [ ]:
experience.explain(method="shap")


## 5. Expliquer les mêmes dossiers avec LIME
Le R² est celui de l’approximation locale ; les poids LIME et SHAP ont des unités différentes. Budget : 2048 perturbations. Les avertissements de calcul restent visibles dans les rapports.

In [ ]:
experience.explain(method="lime")


## 6. Choisir son profil
Utilisateur : facteurs principaux. Métier : synthèse du groupe et éléments à examiner. Technique : comparaison SHAP/LIME. Audit : provenance et contrôles. Les contributions sont calculées une fois et conservées entre profils. Aucun score ne constitue une preuve de fraude.
API native : `helm.explain_profile(X_test, profile="metier", k=10)` ; la préparation reste dans les modules GitHub.


In [ ]:
experience.show()


## 7. Consulter le code et l’audit
Les fichiers GitHub sont présents dans Colab, pas uniquement sur l’ordinateur local. Le module ci-dessous contient les appels HELM réellement exécutés.

In [ ]:
from IPython.display import Code, display
display(Code((SOURCE / "examples/colab/helm_colab/worker.py").read_text(), language="python"))
experience.audit()
